# UMUD Challenge — Submission 6: VGG16-UNet (The DL_Track_US Way)

## Strategy
The competition organiser, Paul Ritsche, is also the author of **DL_Track_US** — the state-of-the-art
published tool for exactly this task. Research reveals it uses a **VGG16 encoder U-Net** with
**task-specific models** (Apo model + Fascicle model trained separately). This notebook mirrors
that exact proven architecture.

**Why this beats our previous attempts:**
- Sub 3 (best, 0.86174): ResNet34-UNet @ 384px — good but generic encoder
- Sub 4 (1.10): EfficientNet-B3 @ 512px — overfit + geometry-warping augmentation
- **Sub 6 (this):** VGG16-UNet @ 384px — the architecture published scientists use for THIS exact task

**Key rules we follow (hard-learned from history):**
1. Separate Apo model and Fascicle model (never mix frames)
2. NO elastic/grid distortion (destroys tissue geometry → Sub 4 disaster)
3. Comma-separated CSV output (not semicolons → Sub 1 disaster)
4. image_id = filename WITH extension (e.g. IMG_0001.tif)
5. Suppress TIFF C++ warnings with `cv2.setLogLevel(0)` + `OPENCV_LOG_LEVEL=SILENT`
6. Scan test folder directly (don't trust sample_submission row count — only had 2 rows!)


In [ ]:
%%capture
!pip install segmentation-models-pytorch albumentations


In [ ]:
import os, sys, math, glob, warnings, logging
# ── Silence EVERYTHING including C++ OpenCV TIFF warnings ──
os.environ["OPENCV_LOG_LEVEL"] = "SILENT"
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

import cv2
cv2.setLogLevel(0)  # kills "WARN:0@xx.xx] global grfmt_tiff.cpp" spam

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm.auto import tqdm

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# ── Paths (matches the exact Kaggle input folder structure shown in the screenshot) ──
INPUT = Path("/kaggle/input")

def _find(root: Path, name: str) -> Path:
    """Walk root until we find a directory whose name matches."""
    for p in root.rglob("*"):
        if p.is_dir() and p.name == name:
            return p
    raise FileNotFoundError(f"{name} not found under {root}")

try:
    COMP = _find(INPUT, "apo_imgs_v1").parent
except FileNotFoundError:
    COMP = INPUT / "umud-challenge-muscle-architecture-in-ultrasound-data"

APO_IMG_DIR   = COMP / "apo_imgs_v1"  / "apo_images_new_model_v1"
APO_MASK_DIR  = COMP / "apo_masks_v1" / "apo_masks_new_model_v1"
FASC_IMG_DIR  = COMP / "fasc_imgs_v1" / "fasc_images_new_model_v1"
FASC_MASK_DIR = COMP / "fasc_masks_v1"/ "fasc_masks_new_model_v1"
TEST_DIR      = COMP / "test_images_v2" / "test_set_v2"
OUT_DIR       = Path("/kaggle/working")

print("\n── Path check ─────────────────────────────")
for name, p in [("APO imgs", APO_IMG_DIR), ("APO masks", APO_MASK_DIR),
                ("FASC imgs", FASC_IMG_DIR), ("FASC masks", FASC_MASK_DIR),
                ("TEST dir", TEST_DIR)]:
    status = "✓" if p.exists() else "✗ MISSING"
    n = len(list(p.rglob("*.*"))) if p.exists() else 0
    print(f"  {status}  {name}: {p}  ({n} files)")

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE           = 384   # Same as our best submission (Sub 3). 512 caused overfitting.
BATCH_SIZE         = 8
EPOCHS             = 25    # With early stopping, we rarely use all 25
PATIENCE           = 6
LR                 = 5e-4
SCALE_MM_PER_PIXEL = 0.06979  # Forensically determined for all 1200x800 images


In [ ]:
# ── VGG16 expects ImageNet normalisation (3-channel input) ───────────────────
_MEAN = (0.485, 0.456, 0.406)
_STD  = (0.229, 0.224, 0.225)

def _gauss_noise():
    """Albumentations API changed var_limit -> std_range. Handle both gracefully."""
    try:
        return A.GaussNoise(var_limit=(5.0, 25.0), p=0.3)
    except TypeError:
        return A.GaussNoise(std_range=(0.02, 0.10), p=0.3)

TRAIN_AUG = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    # Mild affine only – no elastic/grid distortion (destroys tissue geometry!)
    A.ShiftScaleRotate(shift_limit=0.04, scale_limit=0.06, rotate_limit=8,
                       border_mode=cv2.BORDER_CONSTANT, value=0, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.4),
    _gauss_noise(),
    A.Normalize(mean=_MEAN, std=_STD),
    ToTensorV2(),
])

VAL_AUG = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=_MEAN, std=_STD),
    ToTensorV2(),
])

_EXTS = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp"}

class SegDataset(Dataset):
    def __init__(self, img_dir: Path, mask_dir: Path, aug):
        self.aug = aug
        # Build a fast stem→mask lookup (handles name mismatches like _apo, _mask suffixes)
        self._mask_lut = {}
        for p in mask_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in _EXTS:
                clean = p.stem.replace("_apo", "").replace("_fasc", "").replace("_mask", "")
                self._mask_lut[clean] = p

        self.imgs = sorted([p for p in img_dir.rglob("*")
                            if p.is_file() and p.suffix.lower() in _EXTS])
        print(f"  Dataset: {len(self.imgs)} images, {len(self._mask_lut)} masks in lookup")

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        ip = self.imgs[idx]
        clean_stem = ip.stem.replace("_apo", "").replace("_fasc", "").replace("_mask", "")
        mp = self._mask_lut.get(clean_stem, None)

        img = cv2.imread(str(ip), cv2.IMREAD_GRAYSCALE)
        if img is None: img = np.zeros((800, 1200), dtype=np.uint8)

        if mp is not None:
            msk = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
            if msk is None: msk = np.zeros(img.shape[:2], dtype=np.uint8)
            if msk.shape[:2] != img.shape[:2]:            # shape mismatch guard
                msk = cv2.resize(msk, (img.shape[1], img.shape[0]),
                                 interpolation=cv2.INTER_NEAREST)
        else:
            msk = np.zeros(img.shape[:2], dtype=np.uint8)

        msk = (msk > 127).astype(np.uint8)

        # Convert gray→RGB so VGG16 encoder (trained on 3-ch ImageNet) works
        img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        out = self.aug(image=img_rgb, mask=msk)
        return out["image"], out["mask"].float().unsqueeze(0)  # (3,H,W), (1,H,W)


def make_loaders(img_dir, mask_dir):
    full = SegDataset(img_dir, mask_dir, aug=None)
    n = len(full)
    val_n = max(1, int(n * 0.15))
    idx = np.random.RandomState(SEED).permutation(n).tolist()
    train_idx, val_idx = idx[val_n:], idx[:val_n]

    train_ds = SegDataset(img_dir, mask_dir, aug=TRAIN_AUG)
    val_ds   = SegDataset(img_dir, mask_dir, aug=VAL_AUG)
    train_ds.imgs = [full.imgs[i] for i in train_idx]
    val_ds.imgs   = [full.imgs[i] for i in val_idx]
    train_ds._mask_lut = val_ds._mask_lut = full._mask_lut

    tl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=2, pin_memory=True, drop_last=True)
    vl = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=2, pin_memory=True)
    return tl, vl

print("Building Apo loaders…")
apo_tl, apo_vl = make_loaders(APO_IMG_DIR, APO_MASK_DIR)
print("Building Fasc loaders…")
fsc_tl, fsc_vl = make_loaders(FASC_IMG_DIR, FASC_MASK_DIR)
print(f"\nApo  → train={len(apo_tl.dataset)}  val={len(apo_vl.dataset)}")
print(f"Fasc → train={len(fsc_tl.dataset)}  val={len(fsc_vl.dataset)}")


In [ ]:
# ── The DL_Track_US-inspired architecture: VGG16 encoder + U-Net decoder ─────
# smp gives us this in one line. We use 3-channel input (gray→RGB) and 1-channel output.
def build_vgg16_unet():
    return smp.Unet(
        encoder_name    = "vgg16_bn",   # VGG16 with batch norm — same family as DL_Track_US
        encoder_weights = "imagenet",
        in_channels     = 3,            # RGB (converted from grayscale above)
        classes         = 1,            # Single binary mask output
    ).to(DEVICE)

# ── BCE + Dice hybrid loss ─────────────────────────────────────────────────────
class BceDiceLoss(nn.Module):
    def __init__(self, bce_w=0.5, smooth=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_w, self.smooth = bce_w, smooth

    def forward(self, logits, targets):
        bce = self.bce(logits, targets)
        p = torch.sigmoid(logits)
        inter = (p * targets).sum(dim=(1, 2, 3))
        union = p.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        dice  = 1 - (2 * inter + self.smooth) / (union + self.smooth)
        return self.bce_w * bce + (1 - self.bce_w) * dice.mean()

print("VGG16-UNet builder and BceDiceLoss defined ✓")


In [ ]:
def train_one(tag, train_dl, val_dl):
    model = build_vgg16_unet()
    opt   = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
    crit  = BceDiceLoss()
    scaler = torch.amp.GradScaler("cuda")
    best_dice, patience_cnt = -1.0, 0
    save_path = OUT_DIR / f"{tag}_best.pth"

    print(f"\n{'='*55}")
    print(f"  Training  {tag}  [{EPOCHS} epochs max, patience={PATIENCE}]")
    print(f"{'='*55}")

    for epoch in range(1, EPOCHS + 1):
        # ── Train ──────────────────────────────────────────────
        model.train()
        t_loss = 0.0
        for imgs, masks in train_dl:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast("cuda"):
                loss = crit(model(imgs), masks)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            t_loss += loss.item() * imgs.size(0)
        t_loss /= len(train_dl.dataset)
        sched.step()

        # ── Validate ────────────────────────────────────────────
        model.eval()
        dice_sum = 0.0
        with torch.no_grad():
            for imgs, masks in val_dl:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                with torch.amp.autocast("cuda"):
                    p = torch.sigmoid(model(imgs))
                pred = (p > 0.5).float()
                inter = (pred * masks).sum(dim=(1,2,3))
                union = pred.sum(dim=(1,2,3)) + masks.sum(dim=(1,2,3))
                dice_sum += ((2*inter+1)/(union+1)).mean().item() * imgs.size(0)
        val_dice = dice_sum / len(val_dl.dataset)
        lr_now = opt.param_groups[0]["lr"]

        print(f"  Ep {epoch:02d}  loss={t_loss:.4f}  val_dice={val_dice:.4f}  lr={lr_now:.2e}")

        if val_dice > best_dice:
            best_dice = val_dice
            patience_cnt = 0
            torch.save(model.state_dict(), save_path)
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f"  Early stopping at epoch {epoch} (best dice={best_dice:.4f})")
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))
    model.eval()
    print(f"  {tag} done — best val_dice={best_dice:.4f}  saved→{save_path}")
    return model

apo_model  = train_one("apo_vgg16",  apo_tl, apo_vl)
fasc_model = train_one("fasc_vgg16", fsc_tl, fsc_vl)


In [ ]:
def measure_geometry(apo_mask, fasc_mask, scale=SCALE_MM_PER_PIXEL):
    """
    Extract PA (deg), FL (mm), MT (mm) from binary uint8 masks.
    apo_mask and fasc_mask must be 255-valued uint8 numpy arrays at original image size.
    """
    # ── 1. Pennation Angle via PCA on fascicle fragments ──────────────────────
    n, labels, stats, _ = cv2.connectedComponentsWithStats(fasc_mask, 8)
    angles, wts = [], []
    for i in range(1, n):
        area = stats[i, cv2.CC_STAT_AREA]
        if area < 50: continue
        pts = np.column_stack(np.where(labels == i)).astype(np.float64)
        if len(pts) < 5: continue
        cov = np.cov(pts.T)
        if cov.ndim < 2 or cov.shape != (2, 2): continue
        evals, evecs = np.linalg.eigh(cov)
        v = evecs[:, np.argmax(evals)]
        angles.append(np.degrees(np.arctan2(abs(v[0]), abs(v[1]))))
        wts.append(float(area))
    pa_deg = float(np.average(angles, weights=wts)) if angles else 15.0

    # ── 2. Muscle Thickness from aponeurosis separation ───────────────────────
    na, alabs, astats, _ = cv2.connectedComponentsWithStats(apo_mask, 8)
    mt_mm = 20.0   # anatomical fallback
    if na >= 3:
        top2 = sorted(range(1, na), key=lambda i: astats[i, cv2.CC_STAT_AREA], reverse=True)[:2]
        c1, c2 = top2[0], top2[1]
        y1 = np.mean(np.where(alabs == c1)[0])
        y2 = np.mean(np.where(alabs == c2)[0])
        sup_c, deep_c = (c1, c2) if y1 < y2 else (c2, c1)

        sm = (alabs == sup_c).astype(np.uint8)
        dm = (alabs == deep_c).astype(np.uint8)
        sup_bot  = np.where(sm.any(axis=0),  sm.shape[0] - 1 - np.argmax(sm[::-1], axis=0), np.nan)
        deep_top = np.where(dm.any(axis=0),  np.argmax(dm, axis=0), np.nan)

        thick = deep_top - sup_bot
        valid = ~np.isnan(thick) & (thick > 0)
        if valid.sum() > 50:
            mt_mm = float(np.nanmedian(thick[valid]) * scale)

    # ── 3. Fascicle Length via trigonometry ───────────────────────────────────
    fl_mm = mt_mm / math.sin(math.radians(max(pa_deg, 3.0)))

    return (float(np.clip(pa_deg, 3.0, 45.0)),
            float(np.clip(fl_mm,  20.0, 200.0)),
            float(np.clip(mt_mm,  5.0,  40.0)))

print("measure_geometry() defined ✓")


In [ ]:
# ── Normalisation identical to training ───────────────────────────────────────
_INF_AUG = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

def predict_mask(model, img_bgr_gray, tta=True):
    """Return a uint8 binary mask (0/255) at the original image resolution."""
    h0, w0 = img_bgr_gray.shape[:2]
    rgb = cv2.cvtColor(img_bgr_gray, cv2.COLOR_GRAY2RGB)

    x = _INF_AUG(image=rgb)["image"].unsqueeze(0).to(DEVICE)

    with torch.no_grad(), torch.amp.autocast("cuda"):
        p_orig  = torch.sigmoid(model(x))
        if tta:
            x_flip  = torch.flip(x, dims=[3])
            p_flip  = torch.sigmoid(model(x_flip))
            p_flip  = torch.flip(p_flip, dims=[3])
            p_avg   = (p_orig + p_flip) / 2.0
        else:
            p_avg = p_orig

    prob = p_avg.squeeze().cpu().float().numpy()          # (H, W) 0-1
    prob_full = cv2.resize(prob, (w0, h0), interpolation=cv2.INTER_LINEAR)
    mask = (prob_full > 0.5).astype(np.uint8) * 255
    # Small morphological opening to remove speckle noise from predicted mask
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
    return mask


# ── Scan TEST_DIR directly — never trust sample_submission row count! ──────────
test_files = sorted([p for p in TEST_DIR.rglob("*")
                     if p.is_file() and p.suffix.lower() in _EXTS])
print(f"Found {len(test_files)} test images in {TEST_DIR}")

results = []
apo_model.eval(); fasc_model.eval()

for tp in tqdm(test_files, desc="Inference"):
    img = cv2.imread(str(tp), cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"  WARN: could not read {tp.name}, skipping")
        continue

    apo_mask  = predict_mask(apo_model,  img, tta=True)
    fasc_mask = predict_mask(fasc_model, img, tta=True)
    pa, fl, mt = measure_geometry(apo_mask, fasc_mask)

    # image_id = filename WITH extension (e.g. IMG_0001.tif)
    results.append({"image_id": tp.name,
                    "pa_deg":   round(pa, 3),
                    "fl_mm":    round(fl, 3),
                    "mt_mm":    round(mt, 3)})

sub = pd.DataFrame(results)
out_path = OUT_DIR / "submission.csv"
sub.to_csv(out_path, sep=",", index=False)   # COMMA separator — CRITICAL!

print(f"\nSubmission saved → {out_path}  ({len(sub)} rows)")
print(f"  PA range : {sub.pa_deg.min():.1f} – {sub.pa_deg.max():.1f} deg")
print(f"  FL range : {sub.fl_mm.min():.1f} – {sub.fl_mm.max():.1f} mm")
print(f"  MT range : {sub.mt_mm.min():.1f} – {sub.mt_mm.max():.1f} mm")
print("\nHead:")
print(sub.head())
